# Week 22 Day 03 Learning Notebook

This day notebook is fully executable and aligned to the lesson content.

In [1]:
from datetime import date
import warnings
import numpy as np
import pandas as pd

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    from pandas_datareader import data as pdr
except Exception:
    pdr = None

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.6f}'.format

def _synthetic_price_frame(tickers, start='2018-01-01', periods=900, seed=7):
    idx = pd.bdate_range(start, periods=periods)
    out = pd.DataFrame(index=idx)
    rng = np.random.default_rng(seed)
    for i, ticker in enumerate(tickers):
        drift = 0.0002 + 0.00005 * (i + 1)
        vol = 0.010 + 0.002 * i
        rets = rng.normal(drift, vol, size=len(idx))
        out[ticker] = 100.0 * np.exp(np.cumsum(rets))
    return out

def load_market_prices(tickers, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    tickers = list(tickers)
    if yf is None:
        warnings.warn('yfinance unavailable, using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

    try:
        raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
        if isinstance(raw.columns, pd.MultiIndex):
            if 'Close' in raw.columns.get_level_values(0):
                close = raw['Close']
            else:
                close = raw.xs('Close', axis=1, level=0, drop_level=True)
        else:
            close = raw.rename(columns={raw.columns[0]: tickers[0]})
        close = close.reindex(columns=tickers)
        close = close.dropna(how='all').ffill().dropna()
        if close.empty:
            raise ValueError('empty market data from Yahoo Finance')
        return close
    except Exception as exc:
        warnings.warn(f'Yahoo download failed ({exc}); using synthetic data')
        return _synthetic_price_frame(tickers, start=start)

def load_fred_series(series_id, start='2018-01-01', end=None):
    end = end or date.today().isoformat()
    if pdr is None:
        warnings.warn('pandas_datareader unavailable, using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

    try:
        ser = pdr.DataReader(series_id, 'fred', start, end)[series_id]
        ser = ser.ffill().dropna()
        if ser.empty:
            raise ValueError('empty FRED series')
        return ser
    except Exception as exc:
        warnings.warn(f'FRED download failed ({exc}); using synthetic macro data')
        idx = pd.bdate_range(start, end)
        vals = np.linspace(1.0, 1.2, len(idx)) + np.sin(np.arange(len(idx)) / 25) * 0.02
        return pd.Series(vals, index=idx, name=series_id)

# Week 22 Day 03: Python coding speed drills

## Study Duration
- Planned effort: 6-10 hours/day
- Required minimum: 6 hours (core + required extension); optional deep work extends to 10 hours.

## 6-10 Hour Daily Structure
- **Core Block 1 (45 min):** Reset notation (prices, returns, percentages, symbols, units).
- **Core Block 2 (60 min):** Core formulas and compounding intuition with plain-language explanations.
- **Core Block 3 (45 min):** Hand-calculated solved examples plus common traps.
- **Core Block 4 (60 min):** Python/pandas implementation and output verification.
- **Core Block 5 (30 min):** Practice questions, interview drill, and reflection.
- **Required Extension Block A (60 min):** Re-run the real trading example with one alternate ticker and one stress window.
- **Required Extension Block B (60 min):** Restart kernel and rerun all coding cells end-to-end, then add one extra validation test.
- **Optional Deep Work (0-4 hours):** Expand diagnostics, add edge-case tests, and improve interview-ready explanations.

## Why It Matters in Quant
Build speed and accuracy for core quantitative interview domains.

## Continuity and Handoff
- Previous checkpoint: Week 22 Day 02: Statistics and estimation interview drills
- Previous lesson file: content/week-22/day-02.md
- Today's deliverable: Build a local timed coding drill harness.
- Next handoff target: Week 22 Day 04: SQL and data manipulation interview drills
- Next lesson file: content/week-22/day-04.md

## Theory Concepts

### Concept 1: Array and string patterns
Array and string patterns is a core part of 'Quant interview prep I'. Start with notation discipline: define readiness metrics, scoring weights, and evidence thresholds before making final decisions. Then focus on decision quality, communication rigor, and reproducible evidence by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 2: Complexity awareness
Complexity awareness is a core part of 'Quant interview prep I'. Start with notation discipline: define readiness metrics, scoring weights, and evidence thresholds before making final decisions. Then focus on decision quality, communication rigor, and reproducible evidence by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

### Concept 3: Edge-case handling
Edge-case handling is a core part of 'Quant interview prep I'. Start with notation discipline: define readiness metrics, scoring weights, and evidence thresholds before making final decisions. Then focus on decision quality, communication rigor, and reproducible evidence by pairing at least one formula with one real market example (SPY/QQQ/AAPL or phase-relevant assets), verifying units, and documenting one failure mode that appears in stressed regimes.

## Mathematical Foundations (LaTeX)
### Formula 1: Bayes Update
$$
P(H\mid D)=\frac{P(D\mid H)P(H)}{P(D)}
$$
**Plain-English interpretation:** Evidence-driven belief update.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Translate the metric into a go/no-go decision with explicit thresholds and risk guardrails.

### Formula 2: CAGR
$$
CAGR=\left(\frac{V_T}{V_0}\right)^{1/T}-1
$$
**Plain-English interpretation:** Long-horizon growth target.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Translate the metric into a go/no-go decision with explicit thresholds and risk guardrails.

### Formula 3: Gap
$$
Gap_j=Target_j-Current_j
$$
**Plain-English interpretation:** Remaining improvement workload.
**Notation check:** Identify each symbol and its units before coding this formula.
**Real-world anchor:** Translate the metric into a go/no-go decision with explicit thresholds and risk guardrails.

## Symbol Definitions
| Symbol | Meaning | Units | Example |
| --- | --- | --- | --- |
| $P_t$ | Price at time $t$ | USD/share | $110.50 |
| $r_t$ | Simple return | decimal or % | 0.012 = 1.2% |
| $\mu$ | Expected return | annualized decimal | 0.14 |
| $\sigma$ | Volatility (std. dev.) | annualized decimal | 0.18 |
| $S$ | Readiness score | 0 to 100 scale | 83.4 |
| $EV$ | Expected value | R-multiple | 0.45R |
| $Gap_j$ | Target-current skill gap | score points | 7.5 |

## Real Trading Example
- Instruments: SPY, QQQ, TLT
- Macro overlay (FRED): VIXCLS, TEDRATE
- Suggested window: 2018-01-01 to 2026-03-31
- Stress windows to inspect: 2020-03 to 2020-04, 2022-09 to 2022-12
- Scenario context: live-paper trading under risk committee oversight
- Day objective: Solve 3 medium coding problems with clean test coverage.

Execution narrative:
1. Pull market data from Yahoo Finance and align calendars.
2. Pull listed FRED series and join strictly by release-aware timestamps (no look-ahead).
3. Compute today's formulas and compare calm vs stress-window behavior.
4. Translate outputs into one explicit trade action and one hard risk guardrail.
5. Validate that the decision is consistent with topic 'Python coding speed drills'.

## Step-by-Step Solved Problems
### Solved Problem 1: Expected value
Given:
- Win probability=0.58, gain=1.5R, loss=1R.
Solution:
1. $EV=p\cdot Gain-(1-p)\cdot Loss$.
2. EV = 0.58*1.5 - 0.42*1.0 = 0.45R.
Final answer: Expected value = 0.45R per trade.
Common trap: Using one metric in isolation instead of combining expected value, risk limits, and readiness score.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 2: Readiness score
Given:
- Weights=[0.45,0.35,0.20], scores=[82,87,80].
Solution:
1. $S=\sum_jw_js_j$.
2. S = 0.45*82 + 0.35*87 + 0.20*80 = 83.35.
Final answer: Readiness score = 83.35/100.
Common trap: Using one metric in isolation instead of combining expected value, risk limits, and readiness score.
Interpretation: Write one sentence describing how this result would change a real trading decision.

### Solved Problem 3: CAGR target
Given:
- V0=1.00, VT=1.32, T=3 years.
Solution:
1. $CAGR=(V_T/V_0)^{1/T}-1$.
2. CAGR = (1.32/1.00)^(1/3)-1 = 0.0969.
Final answer: Required CAGR = 9.69%.
Common trap: Using one metric in isolation instead of combining expected value, risk limits, and readiness score.
Interpretation: Write one sentence describing how this result would change a real trading decision.

## Coding Walkthrough
1. Build an explicit data-ingestion layer with timestamp and schema checks.
2. Implement today's objective as reusable functions: Build a local timed coding drill harness.
3. Add validation tests for leakage, NaNs, and unrealistic outliers.
4. Produce diagnostic plots and summarize one actionable trading rule.
5. Record one failure mode and one mitigation in comments.

Reference implementation sketch:
```python
readiness = score_readiness(quiz_scores, mock_scores, project_rubrics)
posterior = bayes_update(prior=0.55, likelihood=0.72, evidence=0.61)
roadmap = build_90_day_plan(readiness, posterior)
export_launch_checklist(roadmap)
```

## Block 5: Practice, Quiz, and Interview Drill

### Practice Problems
1. Re-derive today's formulas manually and define every variable and unit.
2. Re-run the real trading example with one alternate ticker and compare outputs.
3. Stress-test one assumption and write one explicit risk-control rule.
4. Extend the coding walkthrough with one validation test and one edge-case test.
5. Record one interview-ready explanation in less than 60 seconds.

### Daily Quiz (Realistic Interview Style)
1. PM interview question (Week 22 Day 03): Explain Bayes Update and define every symbol clearly for a live-paper trading month with strict risk committee oversight.
   - Model answer: "I use Bayes Update as a decision bridge from market observations to position sizing. The formula is $P(H\mid D)=\frac{P(D\mid H)P(H)}{P(D)}$. I define each symbol with units first, then compute one concrete value, and finally state what trade action changes because of the result in this regime."
2. Risk manager question: Using one real ticker from this lesson, what hard guardrail would you enforce before live deployment?
   - Model answer: "I would run the workflow on SPY and a stress-sensitive peer, then halt promotions when daily max drawdown breaches the approved loss budget. If the guardrail triggers, I switch to paper-trade monitoring and block new risk until diagnostics re-pass."
3. Data integrity question: Which checks must pass before you trust the output and place risk?
   - Model answer: "Before trading I verify broker feed latency checks, execution timestamp reconciliation, and policy-rule audit logs. If any check fails, I classify the run as non-tradable and log the incident."
4. Production question: Why does 'Python coding speed drills' matter in live trading systems?
   - Model answer: "Python coding speed drills matters because deployment decisions must combine edge estimates with operational and risk controls. In production I need reproducible calculations, explicit control limits, and escalation rules that survive stress windows."
5. Decision question: If your key metric degrades for three consecutive sessions, what is your fallback plan?
   - Model answer: "I freeze promotions, continue paper-trade only, and escalate to committee with replay evidence. I only restore risk after rerun evidence confirms that assumptions are stable again."

Scoring rubric:
- 10/10: correct notation, one numeric example, explicit guardrail, data checks, and escalation path.
- 8/10: mostly correct notation plus a clear guardrail and fallback action.
- 6/10: partial correctness but vague controls or missing data validation.
- Below 6/10: formula recall without decision-quality risk controls.

### Interview Drill
- Prompt: "Walk me through Python coding speed drills in a launch readiness panel before promoting from paper-trade."
- What interviewers look for:
  1. Correct notation and units.
  2. Ability to connect theory to a real trade decision under constraints.
  3. Awareness of edge cases, costs, and failure modes.
  4. Clear escalation rule when guardrails are breached.
- Model answer framework:
  - Context: define business objective and market regime.
  - Method: state formula, assumptions, and validation checks clearly.
  - Decision: explain one actionable rule, one risk guardrail, and one fallback action.

## Required Extension Track (2+ Hours)
- Re-run today's notebook from a fresh kernel so outputs are reproducible without hidden state.
- Add one additional risk guardrail and verify how it changes trade/no-trade decisions.
- Document one failure mode, one mitigation, and one escalation rule for production use.

Extension completion checks:
- [ ] Notebook restarted and all coding cells rerun successfully
- [ ] At least one extra validation/edge-case test added
- [ ] Risk guardrail and fallback action documented

## Reflection Question
Which coding pattern causes most debugging delays?

## Completion Checklist
- [ ] Formula derivations re-worked manually
- [ ] Real trading example reproduced with data checks
- [ ] Solved problems reviewed and understood
- [ ] Coding walkthrough executed and verified
- [ ] Full notebook rerun completed from clean kernel
- [ ] Reflection logged in progress tracker


## Runnable Day Example
Run this example for Week 22 Day 03, inspect outputs, then complete the quiz.

In [2]:
# Integration demo: readiness tracker anchored to market context
np.random.seed(2203)
spy = load_market_prices(['SPY'], start='2018-01-01')['SPY']
ret = spy.pct_change().dropna()
macro_unrate = load_fred_series('UNRATE', start='2018-01-01').dropna()

ann_return = float((1 + ret).prod() ** (252 / len(ret)) - 1)
ann_vol = float(ret.std() * np.sqrt(252))
sharpe = float((ann_return - 0.03) / max(ann_vol, 1e-8))

quiz_avg = float(np.clip(70 + 10 * sharpe, 60, 95))
mock_scores = np.clip(np.array([72, 78, 81]) + sharpe * 5, 60, 95)
macro_level = float(macro_unrate.iloc[-1]) if not macro_unrate.empty else float('nan')

{
    'spy_annualized_return': ann_return,
    'spy_annualized_vol': ann_vol,
    'spy_sharpe_proxy': sharpe,
    'latest_unemployment_rate': macro_level,
    'quiz_average': round(quiz_avg, 2),
    'mock_scores': [float(round(x, 2)) for x in mock_scores],
}


/var/folders/7l/31hylb_513bbgfbz8nnsgp840000gn/T/ipykernel_74409/2091198247.py:58: UserWarning: pandas_datareader unavailable, using synthetic macro data
  warnings.warn('pandas_datareader unavailable, using synthetic macro data')


{'spy_annualized_return': 0.14223480916595577,
 'spy_annualized_vol': 0.19307141078948611,
 'spy_sharpe_proxy': 0.5813124206583341,
 'latest_unemployment_rate': 1.1802755958569258,
 'quiz_average': 75.81,
 'mock_scores': [74.91, 80.91, 83.91]}

## ReAct Verification Cell
Use this execution cell to validate one trade decision with an explicit risk guardrail.

In [3]:
# ReAct-style verification: observe -> reason -> act -> verify
np.random.seed(11203)

observe_tickers = ['SPY', 'QQQ', 'TLT']
observe_prices = load_market_prices(observe_tickers, start='2020-01-01')
observe_returns = observe_prices.pct_change().dropna()

if observe_returns.empty:
    raise ValueError('No returns available for verification checks')

ann_vol = float(observe_returns['SPY'].std() * np.sqrt(252))
ann_ret = float((1 + observe_returns['SPY']).prod() ** (252 / len(observe_returns)) - 1)
sharpe_proxy = float((ann_ret - 0.03) / max(ann_vol, 1e-8))

# Risk-first deployment gate used in realistic interview responses.
guardrail = 'de-risk' if ann_vol > 0.30 else 'monitor'
decision = 'deploy-paper-trade' if sharpe_proxy > 0.40 and guardrail == 'monitor' else 'hold-and-review'

verification = {
    'topic': 'Python coding speed drills',
    'week': 22,
    'day': 3,
    'observe_annual_return': ann_ret,
    'observe_annual_vol': ann_vol,
    'reason_sharpe_proxy': sharpe_proxy,
    'act_guardrail': guardrail,
    'verify_decision': decision,
}

verification


{'topic': 'Python coding speed drills',
 'week': 22,
 'day': 3,
 'observe_annual_return': 0.14923382208457214,
 'observe_annual_vol': 0.20501282703947718,
 'reason_sharpe_proxy': 0.5815920096629492,
 'act_guardrail': 'monitor',
 'verify_decision': 'deploy-paper-trade'}

## Week 22 Day 03 Quiz

Topic: **Python coding speed drills**

Real-world interview questions (answer first, then run the next cell for model answers):
1. PM question: Explain Expected Value (EV = p * Gain - (1-p) * Loss) and define every symbol clearly.
2. Risk question: What hard guardrail would you enforce before deployment?
3. Data question: Which validation checks must pass before trusting this output?
4. Production question: Why does this topic matter in live systems, not just in notebooks?
5. Escalation question: If the metric degrades for three sessions, what is your fallback action?

Hint for Q2: halt promotions when max drawdown breaches deployment policy.
Hint for Q3: execution timestamp reconciliation and policy logging
Scoring: full credit requires notation correctness, one numeric example, explicit guardrail, and fallback path.

In [4]:
# Quiz model answers (auto-generated)
p = 0.58
gain = 1.4
loss = 1.0
metric_value = p * gain - (1 - p) * loss
metric_name = 'expected_value'

print('Interview Question 1 (model answer):')
print('  Formula focus: Expected Value')
print('  Formula text : EV = p * Gain - (1-p) * Loss')
print('  Scenario     : live paper-trade month')
print('  Numeric value:', round(float(metric_value), 6))

print('\nInterview Question 2 (model answer):')
print('  For a real ticker like SPY, I would enforce this guardrail before deployment:')
print('  halt promotions when max drawdown breaches policy budget.')

print('\nInterview Question 3 (model answer):')
print('  I validate timestamps, missing data handling, and assumption consistency before taking risk.')

print('\nInterview Question 4 (model answer):')
print('  Topic:', 'Python coding speed drills')
print('  This matters because production systems need reproducible metrics, explicit controls,')
print('  and a fallback decision path when stress conditions invalidate baseline assumptions.')

print('\nInterview Question 5 (model answer):')
print('  Fallback action: freeze promotion and escalate to committee with replay evidence.')

print('\nNumeric verification:')
print('  metric_name  =', metric_name)
print('  metric_value =', round(float(metric_value), 6))


Interview Question 1 (model answer):
  Formula focus: Expected Value
  Formula text : EV = p * Gain - (1-p) * Loss
  Scenario     : live paper-trade month
  Numeric value: 0.392

Interview Question 2 (model answer):
  For a real ticker like SPY, I would enforce this guardrail before deployment:
  halt promotions when max drawdown breaches policy budget.

Interview Question 3 (model answer):
  I validate timestamps, missing data handling, and assumption consistency before taking risk.

Interview Question 4 (model answer):
  Topic: Python coding speed drills
  This matters because production systems need reproducible metrics, explicit controls,
  and a fallback decision path when stress conditions invalidate baseline assumptions.

Interview Question 5 (model answer):
  Fallback action: freeze promotion and escalate to committee with replay evidence.

Numeric verification:
  metric_name  = expected_value
  metric_value = 0.392
